**Lab type:** debug

**Course:** ML101 — Intro to Machine Learning

**Lesson:** Decision Trees and Ensemble Methods

**Task:** The AI-generated analysis below contains 3 bugs. For each bug: identify what is wrong, explain why the output is misleading, and write the corrected code in the fix cell.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

np.random.seed(42)
n = 500
age = np.random.randint(22, 65, n)
income = np.random.normal(60000, 20000, n).clip(20000, 200000)
credit_score = np.random.randint(300, 850, n)
employment = np.random.choice(['employed', 'self-employed', 'unemployed'], n, p=[0.6, 0.25, 0.15])
approved = ((credit_score > 650) & (income > 40000) & (employment != 'unemployed')).astype(int)
# Add noise
flip_idx = np.random.choice(n, 40, replace=False)
approved[flip_idx] = 1 - approved[flip_idx]
df = pd.DataFrame({
    'age': age,
    'income': income,
    'credit_score': credit_score,
    'employment': employment,
    'approved': approved
})
print(df.head())
print(f"\nApproval rate: {df['approved'].mean():.2f}")

## Step 1: Train a Decision Tree Classifier

The analyst encodes the categorical feature, splits the data, and trains a decision tree to predict loan approval.

In [ ]:
# AI-generated — contains Bug 1
X_encoded = df.copy()
le = LabelEncoder()
X_encoded['employment'] = le.fit_transform(X_encoded['employment'])
X = X_encoded[['age', 'income', 'credit_score', 'employment']]
y = X_encoded['approved']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt = DecisionTreeClassifier(random_state=42)  # <- Bug 1: no max_depth
dt.fit(X_train, y_train)
print(f"Train accuracy: {accuracy_score(y_train, dt.predict(X_train)):.3f}")
print(f"Test accuracy:  {accuracy_score(y_test, dt.predict(X_test)):.3f}")

**Bug 1 Investigation:** Run the cell above. Compare training accuracy to test accuracy. What does the large gap tell you about the model? What parameter is missing?

In [ ]:
# Fix Bug 1 here
# Set max_depth=5 to prevent the tree from memorising the training data
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Step 2: Encoding Categorical Features

The analyst now encodes the employment column before splitting the dataset into train and test sets.

In [ ]:
# AI-generated — contains Bug 2
df2 = df.copy()
le2 = LabelEncoder()
df2['employment'] = le2.fit_transform(df2['employment'])  # <- Bug 2: full column encoded before split

X2 = df2[['age', 'income', 'credit_score', 'employment']]
y2 = df2['approved']
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X2, y2, test_size=0.2, random_state=42)
print(f"Encoding mapping: {dict(zip(le2.classes_, le2.transform(le2.classes_)))}")

**Bug 2 Investigation:** Run the cell above. At what point is `LabelEncoder.fit` called relative to the train/test split? Why is this a problem in production or cross-validation?

In [ ]:
# Fix Bug 2 here
# Split first, then encode only from training data
# Hint: use pd.get_dummies or OrdinalEncoder fitted only on X_train
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Step 3: Interpreting Feature Importances

The analyst trains a Random Forest and reads off feature importances to determine which features drive loan approval.

In [ ]:
# AI-generated — contains Bug 3
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr2, y_tr2)
importances = dict(zip(X2.columns, rf.feature_importances_))
print("Feature importances:")
for feat, imp in sorted(importances.items(), key=lambda x: -x[1]):
    print(f"  {feat}: {imp:.3f}")
# Comment: "employment is the most important feature"  <- Bug 3

**Bug 3 Investigation:** Run the cell above. How was `employment` encoded in Step 2? How does label encoding affect the importance score reported by a tree-based model? Why is the analyst's conclusion unreliable?

In [ ]:
# Fix Bug 3 here
# Use one-hot encoding so each employment category gets its own column,
# making per-category importances interpretable
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Corrected Analysis

The cell below applies all three fixes: depth-limited decision tree, split-first encoding, and one-hot encoding for interpretable feature importances.

In [ ]:
# --- Fix 1 + 2: split first, then encode using training data only ---
df_c = df.copy()
X_c = df_c[['age', 'income', 'credit_score', 'employment']]
y_c = df_c['approved']

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

# Fix 3: one-hot encode categorical feature
X_tr_ohe = pd.get_dummies(X_tr_c, columns=['employment'])
X_te_ohe = pd.get_dummies(X_te_c, columns=['employment'])
X_te_ohe = X_te_ohe.reindex(columns=X_tr_ohe.columns, fill_value=0)

# Fix 1: limit tree depth to avoid overfitting
dt_c = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_c.fit(X_tr_ohe, y_tr_c)
print(f"Train accuracy: {accuracy_score(y_tr_c, dt_c.predict(X_tr_ohe)):.3f}")
print(f"Test accuracy:  {accuracy_score(y_te_c, dt_c.predict(X_te_ohe)):.3f}")

# Fix 3: interpretable feature importances with one-hot columns
rf_c = RandomForestClassifier(n_estimators=100, random_state=42)
rf_c.fit(X_tr_ohe, y_tr_c)
importances_c = dict(zip(X_tr_ohe.columns, rf_c.feature_importances_))
print("\nCorrected feature importances:")
for feat, imp in sorted(importances_c.items(), key=lambda x: -x[1]):
    print(f"  {feat}: {imp:.3f}")